In [ ]:
!pip install einops
!pip install vit_pytorch

In [ ]:
models_save_path = "/content/drive/MyDrive/Tirocinio/Finale/Models/"
results_save_path_agent = "/content/drive/MyDrive/Tirocinio/Finale/Results/"
dataset_path = "/content/drive/MyDrive/Tirocinio/Transformers/Datasets/"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd

# Libraries for image manipulation and machine learning
import torchvision
import torchvision.transforms as transforms
import torch
import torch.utils.data as data
import torch.nn as nn
from torch.nn import functional
import torch.optim as optim

# Libraries for model evaluation metrics
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Libraries for tensor manipulation
from einops import rearrange
from einops.layers.torch import Rearrange

# Libraries for simulation environments (gym)
import gym

# Libraries for data visualization
import matplotlib.pyplot as plt

# Other common libraries
from collections import namedtuple
import math
import time
import os
import random

In [ ]:
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

In [ ]:
def pair(t):
    return t if isinstance(t, tuple) else (t, t)

def posemb_sincos_2d(patches, temperature = 10000, dtype = torch.float32):
    _, h, w, dim, device, dtype = *patches.shape, patches.device, patches.dtype

    y, x = torch.meshgrid(torch.arange(h, device = device), torch.arange(w, device = device), indexing = 'ij')
    assert (dim % 4) == 0, 'feature dimension must be multiple of 4 for sincos emb'
    omega = torch.arange(dim // 4, device = device) / (dim // 4 - 1)
    omega = 1. / (temperature ** omega)

    y = y.flatten()[:, None] * omega[None, :]
    x = x.flatten()[:, None] * omega[None, :]
    pe = torch.cat((x.sin(), x.cos(), y.sin(), y.cos()), dim = 1)
    return pe.type(dtype)

# classes

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )
    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64):
        super().__init__()
        inner_dim = dim_head *  heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.norm = nn.LayerNorm(dim)

        self.attend = nn.Softmax(dim = -1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)
        self.to_out = nn.Linear(inner_dim, dim, bias = False)

    def forward(self, x):
        x = self.norm(x)

        qkv = self.to_qkv(x).chunk(3, dim = -1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        attn = self.attend(dots)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Attention(dim, heads = heads, dim_head = dim_head),
                FeedForward(dim, mlp_dim)
            ]))
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x

class SimpleAgentViT(nn.Module):
    def __init__(self, *, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, channels = 3, dim_head = 64):
        super().__init__()

        self.patches = []
        image_height, image_width = pair(image_size)
        patch_height, patch_width = pair(patch_size)

        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'

        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width

        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b h w (p1 p2 c)', p1 = patch_height, p2 = patch_width),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim),
            nn.LayerNorm(dim),
        )

        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim)

        self.to_latent = nn.Identity()
        self.linear_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, img):
        *_, h, w, dtype = *img.shape, img.dtype

        x = self.to_patch_embedding(img)
        pe = posemb_sincos_2d(x)
        x = rearrange(x, 'b ... d -> b (...) d') + pe

        # Masking the input based on selected patches
        mask = torch.tensor(self.patches, dtype=torch.bool)
        x = x[:, mask, :]

        x = self.transformer(x)
        x = x.mean(dim = 1)

        x = self.to_latent(x)
        return self.linear_head(x)

    def set_patches(self, mask):
        """Accept a pre-computed binary mask (0.0 / 1.0) directly.
        No thresholding — caller must pass a topk_mask() result."""
        if isinstance(mask, np.ndarray):
            model_device = next(self.parameters()).device
            mask = torch.tensor(mask, dtype=torch.float32, device=model_device)
        elif not isinstance(mask, torch.Tensor):
            model_device = next(self.parameters()).device
            mask = torch.tensor(mask, dtype=torch.float32, device=model_device)
        self.patches = mask.tolist()

    def get_patches(self):
        return self.patches

    def get_att(self, img):
        *_, h, w, dtype = *img.shape, img.dtype

        x = self.to_patch_embedding(img)
        pe = posemb_sincos_2d(x)
        x = rearrange(x, 'b ... d -> b (...) d') + pe

        attn, ff = self.transformer.layers[0]
        x = attn(x)
        return x

# ── Global Top-K mask helper ─────────────────────────────────────────
# Single source of truth for patch selection used everywhere:
#   training, validation, testing, evaluation, visualisation.
def topk_mask(q_values: torch.Tensor, K: int) -> torch.Tensor:
    """Return a float binary mask with exactly K ones at the top-K positions.

    Args:
        q_values : 1-D tensor of Q-values, shape (n_patches,)
        K        : number of patches to select

    Returns:
        mask : float32 tensor of shape (n_patches,) with K ones, rest zeros
    """
    mask = torch.zeros_like(q_values, dtype=torch.float32)
    _, idx = torch.topk(q_values, K)
    mask[idx] = 1.0
    return mask

In [ ]:
batch_size = 256
img_size = 32
dataset_name = "MNIST_"

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

transform_validation = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# Prepare/download dataset
trainset = torchvision.datasets.MNIST(root=dataset_path, train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

validationset = torchvision.datasets.MNIST(root=dataset_path, train=False, download=True, transform=transform_validation)

dataset_size = len(validationset)
validation_size = int(0.95 * dataset_size)
test_size = dataset_size - validation_size

validationset, testset = data.random_split(validationset, [validation_size, test_size])

validation_loader = torch.utils.data.DataLoader(validationset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

classes = [str(i) for i in range(10)]
print(classes)

In [ ]:
def train_iter_agent(model, optimizer, data, target):
    start_time = time.time()

    model.train()

    optimizer.zero_grad()
    out = functional.log_softmax(model(data), dim=1)
    loss = functional.nll_loss(out, target)
    loss.backward()
    optimizer.step()

    iteration_time = time.time() - start_time

    return loss.item(), iteration_time

In [ ]:
def evaluate_agent(model, data_load, device):
    """Evaluate ViTnet using DDQN top-K patch selection (consistent with training).

    NOTE: This function is kept for backward compatibility but is superseded
    by TrainingTestingAgent.evaluate_epoch() during the RL training loop.
    """
    # Use the globally defined topk_mask — same policy as training
    n_patches = len(model.get_patches()) if model.get_patches() else total_patches

    model.eval()

    elements = 0
    csamp = 0
    tloss = 0

    with torch.no_grad():
        for data, target in data_load:

            elements += len(data)
            data = data.to(device)
            target = target.to(device)

            output = functional.log_softmax(model(data), dim=1)
            loss = functional.nll_loss(output, target, reduction='sum')
            _, pred = torch.max(output, dim=1)

            tloss += loss.item()
            csamp += pred.eq(target).sum()

    loss_val = tloss / elements
    acc_val = (100.0 * csamp / elements).cpu()

    print('\nAverage test loss: ' + '{:.4f}'.format(loss_val) +
          '  Accuracy:' + '{:5}'.format(csamp) + '/' +
          '{:5}'.format(elements) + ' (' +
          '{:4.2f}'.format(acc_val) + '%)\n')

    return loss_val, acc_val

In [ ]:
def train_validation_agent(model, optimizer, train_data, train_target, validation_loader, device):

  start_time = time.time()

  tr_loss = train_iter_agent(model, optimizer, train_data, train_target)

  val_loss, val_acc = evaluate_agent(model, validation_loader, device)

  iteration_time = time.time() - start_time

  return tr_loss, val_loss, val_acc, iteration_time

In [ ]:
class MultiContinue():

    def __init__(self,  n_patch, device):
        self.n_patch = n_patch
        self.device = device

    def sample(self):
        action = np.random.rand(self.n_patch)
        return torch.tensor(action, device=self.device, dtype=torch.float)



class ViTEnv(gym.Env):
    def __init__(self, ViTnet, n_patch, optimizer, loss_weight, time_weight, device, n_patch_selected = 1, seed = None):
        super().__init__()

        self.ViTnet = ViTnet
        self.optimizer = optimizer

        self.seed = seed

        self.loss_weight = loss_weight
        self.time_weight = time_weight

        self.action_space = MultiContinue(n_patch, device)

        self.device = device

        self.train_loss_list = []
        self.train_time_list = []

    def step_train(self, action, train_data, train_target):
        action_tensor = torch.tensor(action, device=self.device, dtype=torch.float32)
        mask = topk_mask(action_tensor, n_patch_selected)
        self.ViTnet.set_patches(mask)

        train_iter_agent(self.ViTnet, self.optimizer, train_data, train_target)

    def step_reward(self, action, train_data, train_target):
        action_tensor = torch.tensor(action, device=self.device, dtype=torch.float32)
        mask = topk_mask(action_tensor, n_patch_selected)
        self.ViTnet.set_patches(mask)

        current_patches = self.ViTnet.get_patches()

        print(f'  Patch list: {current_patches}')
        print(f'  Selected Patches: {current_patches.count(1)}')

        reward = self.get_reward(current_patches, train_data = train_data, train_target = train_target)

        return self.get_state(train_data), reward


    def get_reward(self, action, train_data, train_target):

        train_loss, iteration_time = train_iter_agent(self.ViTnet, self.optimizer, train_data, train_target)

        self.train_time_list.append(iteration_time)
        self.train_loss_list.append(train_loss)

        num_zeros = action.count(0)
        ideal_zeros = len(action) - n_patch_selected;
        patches_reward = (- abs(num_zeros - ideal_zeros) / ideal_zeros)
        loss_reward = (self.train_loss_list[0]/train_loss)
        print(f'  loss_reward: {loss_reward}')
        print(f'  patches_reward: {patches_reward}')
        reward = loss_reward * self.loss_weight + patches_reward * self.time_weight

        return reward

    def get_state(self, data):
        with torch.no_grad():
            return self.ViTnet.get_att(data)

In [ ]:
Transition = namedtuple('Transition',
                        ('state', 'new_state', 'reward'))


class ReplayBuffer(object):
    def __init__(self, capacity, batch_size):
        self.batch_size = batch_size
        self.memory = []
        self.capacity = capacity

    def push(self, *args):
        if len(self.memory) >= self.capacity:
            index_to_remove = random.randint(0, len(self.memory) - 1)
            self.memory.pop(index_to_remove)
        self.memory.append(Transition(*args))

    def sample(self):
        return random.sample(self.memory, self.batch_size)

    def __len__(self):
        return len(self.memory)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, input_features, output_features):
        super(QNetwork, self).__init__()

        self.fc_layers = nn.Sequential(
            nn.Linear(input_features, 1024),
            nn.ReLU(),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, output_features)
        )


    def forward(self, input):
        x = self.fc_layers(input)
        return x

In [ ]:
class CMABAgent(nn.Module):
    def __init__(self, att_dim, n_patches):
        super().__init__()
        self.policy = nn.Linear(att_dim, 1)

        self.n_patches = n_patches

    def forward(self, state):
        # state: (B, n_patches, att_dim)

        logits = self.policy(state).squeeze(-1)  # (B, n_patches)
        probs = torch.sigmoid(logits)

        actions = (probs > 0.5).float()

        return actions, probs

In [ ]:
import torch.optim as optim

class DQNAgent():

    def __init__(self, buffer_batch_size, att_dim, n_patches, buffer_size, gamma, tau, update_every, lr, env, device):
        self.device = device
        self.gamma = gamma
        self.tau = tau
        self.update_every = update_every
        self.buffer_batch_size = buffer_batch_size
        self.env = env

        # QNetwork input: n_patches * att_dim, output: n_patches
        q_network_input_features = n_patches * att_dim
        q_network_output_features = n_patches

        self.policy_net = QNetwork(q_network_input_features, q_network_output_features).to(device)
        self.target_net = QNetwork(q_network_input_features, q_network_output_features).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer(buffer_size, buffer_batch_size)
        self.steps_done = 0

    def select_action(self, state, eps):
        self.steps_done += 1
        if random.random() > eps:
            with torch.no_grad():
                # state: (batch_size, n_patches, att_dim) -> mean over image batch -> (1, n_patches, att_dim)
                state_mean = state.mean(dim=0, keepdim=True).to(self.device)  # (1, n_patches, att_dim)
                state_flat = state_mean.view(1, -1)                           # (1, n_patches * att_dim)
                q_values = self.policy_net(state_flat)                        # (1, n_patches)
                return q_values.squeeze(0).cpu().numpy()                      # (n_patches,)
        else:
            return self.env.action_space.sample().cpu().numpy()

    def optimize_model(self):
        """Double DQN update.

        Standard DQN computes the target as:
            y = r + gamma * max_a target_net(s')[a]
        which suffers from maximisation bias because the same network
        selects AND evaluates the next action.

        Double DQN decouples the two steps:
            a* = argmax_a  policy_net(s')[a]   # action SELECTION by policy net
            y  = r + gamma * target_net(s')[a*] # action EVALUATION by target net
        This removes the upward bias and stabilises training.

        In this contextual-bandit variant the state is a per-patch attention
        embedding and each patch dimension is treated as an independent
        'action-value'. The DDQN correction is applied patch-wise:
            a*[p]  = argmax over the two networks for patch p
            target = r + gamma * target_net(s')[p, a*[p]]
        """
        if len(self.memory) < self.buffer_batch_size:
            return

        transitions = self.memory.sample()
        batch = Transition(*zip(*transitions))

        # ── Batch tensors ────────────────────────────────────────────────
        # state     shape: (B, n_patches, att_dim)  [after torch.cat]
        # new_state shape: (B, n_patches, att_dim)
        # reward    shape: (B,)  — scalar reward per transition
        state_batch     = torch.cat(batch.state).to(self.device)      # (B, n_patches*att_dim) after view
        new_state_batch = torch.cat(batch.new_state).to(self.device)
        reward_batch    = torch.cat(batch.reward).to(self.device)     # (B,)

        B = self.buffer_batch_size

        state_flat     = state_batch.view(B, -1)      # (B, n_patches * att_dim)
        new_state_flat = new_state_batch.view(B, -1)  # (B, n_patches * att_dim)

        # ── Current Q-values from policy net ────────────────────────────
        # Q_policy[b, p] = estimated value of patch p in state s_b
        Q_policy = self.policy_net(state_flat)        # (B, n_patches)

        # ── DDQN target computation ──────────────────────────────────────
        with torch.no_grad():
            # Step 1 — action selection: which patch has highest Q in s'?
            #          Use POLICY net to avoid maximisation bias.
            Q_next_policy = self.policy_net(new_state_flat)   # (B, n_patches)
            best_actions  = Q_next_policy.argmax(dim=1)       # (B,)  best patch index per sample

            # Step 2 — action evaluation: value of that patch in s'
            #          Use TARGET net (frozen weights) for evaluation.
            Q_next_target = self.target_net(new_state_flat)   # (B, n_patches)
            # Gather the target-net Q-value for each sample's best action
            Q_next_best   = Q_next_target.gather(
                1, best_actions.unsqueeze(1)
            ).squeeze(1)                                       # (B,)

            # Bellman target, broadcast to all patch dimensions
            # reward_batch: (B,)  →  target: (B, n_patches)
            target = (reward_batch + self.gamma * Q_next_best).unsqueeze(1).expand_as(Q_policy)

        # ── Huber loss + gradient clip ───────────────────────────────────
        criterion = nn.SmoothL1Loss()
        loss = criterion(Q_policy, target)

        self.optimizer.zero_grad()
        loss.backward()
        for param in self.policy_net.parameters():
            param.grad.data.clamp_(-1, 1)
        self.optimizer.step()

        # ── Periodic hard update of target network ───────────────────────
        if self.steps_done % self.update_every == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())


In [ ]:
class TrainingTestingAgent():

    def __init__(self, buffer_batch_size, get_reward_every, batch_size, model, att_dim, n_patches, epochs, env, buffer_size, gamma, tau, update_every, lr, eps_end, eps_start, eps_decay, train_loader, validation_loader, device):

      self.env = env

      self.epochs = epochs
      self.eps_start = eps_start
      self.eps_end = eps_end
      self.eps_decay = eps_decay
      self.get_reward_every = get_reward_every

      self.batch_size = batch_size

      self.validation_acc = []
      self.validation_loss = []
      self.epoch_time_list = []
      self.validation_precision = []
      self.validation_recall = []
      self.validation_f1 = []

      self.ViTnet = model

      self.n_patches = n_patches

      self.agent = DQNAgent(buffer_batch_size, att_dim, n_patches, buffer_size, gamma, tau, update_every, lr, self.env, device)

      self.train_loader = train_loader
      self.validation_loader = validation_loader

      self.device = device


    def train_test(self):

      step_reward = []
      selected_patch_list = []
      epoch = 0
      iteration = 0
      eps = self.eps_start

      while epoch < self.epochs:

          epoch += 1
          print(f'Epoch: {epoch}/{self.epochs}')

          start_time = time.time()
          samples = len(self.train_loader.dataset)

          for i, (data, target) in enumerate(self.train_loader):
              i += 1

              iteration += 1

              data = data.to(self.device)
              target = target.to(self.device)

              state = self.env.get_state(data)  # (batch_size, n_patches, att_dim)

              patch_list = self.agent.select_action(state, eps)

              selected_patch_list.append(patch_list)

              # Convert raw Q-values → Top-K binary mask (consistent policy)
              action_mask = topk_mask(
                  torch.tensor(patch_list, dtype=torch.float32), n_patch_selected
              )

              if i % self.get_reward_every != 0:
                self.env.step_train(action_mask.numpy(), data, target)

              else:
                new_state, reward = self.env.step_reward(action_mask.numpy(), data, target)

                print(f'  Epsilon: {eps},   Reward: {reward}')

                step_reward.append(reward)

                if epoch != 1:
                  # ----------------------------------------------------------------
                  # FIX: mean-pool over the image-batch dimension before storing.
                  # state / new_state shape: (batch_size, n_patches, att_dim)
                  # After mean:              (1,          n_patches, att_dim)
                  # After torch.cat(8 items):(8,          n_patches, att_dim)  <-- correct
                  # view(8, -1):             (8, n_patches * att_dim)          <-- matches QNetwork
                  # ----------------------------------------------------------------
                  state_stored     = state.mean(dim=0, keepdim=True).detach().cpu()
                  new_state_stored = new_state.mean(dim=0, keepdim=True).detach().cpu()
                  reward_stored    = torch.tensor([reward], dtype=torch.float32)

                  self.agent.memory.push(state_stored, new_state_stored, reward_stored)

                  self.agent.optimize_model()

              eps = self.eps_end + (self.eps_start - self.eps_end) * math.exp(-1. * iteration / self.eps_decay)

          print(f'\nInizio Testing')
          loss, acc, precision, recall, f1 = self.evaluate_epoch(self.validation_loader, models_save_path, dataset_name, self.device)

          epoch_time = time.time()-start_time

          self.validation_acc.append(acc)
          self.validation_loss.append(loss)
          self.validation_precision.append(precision)
          self.validation_recall.append(recall)
          self.validation_f1.append(f1)
          self.epoch_time_list.append(epoch_time)

          print(f'Epoch time: {epoch_time}')
          print("#"*40)

          print("#"*40)
          print('Episode End')
          print("#"*40)
          print("#"*40)

      return step_reward, selected_patch_list



    def evaluate_epoch(self, data_load, models_save_path, dataset_name, device):
        """Evaluate using the same Top-K DDQN policy as training (no all-ones mask)."""
        self.ViTnet.eval()

        elements = 0
        csamp = 0
        tloss = 0
        all_predictions = []
        all_targets = []

        with torch.no_grad():
            for data, target in data_load:

                elements += len(data)
                data = data.to(device)
                target = target.to(device)

                # ── CONSISTENT POLICY: same Top-K selection as training ──
                state      = self.env.get_state(data)                          # (B, n_patches, att_dim)
                state_mean = state.mean(dim=0, keepdim=True).to(self.device)   # (1, n_patches, att_dim)
                state_flat = state_mean.view(1, -1)                            # (1, n_patches * att_dim)
                q_values   = self.agent.policy_net(state_flat).squeeze(0)      # (n_patches,)
                mask       = topk_mask(q_values, n_patch_selected)
                self.ViTnet.set_patches(mask)
                # ────────────────────────────────────────────────────────

                predictions = self.ViTnet(data)

                output = functional.log_softmax(predictions, dim=1)
                loss = functional.nll_loss(output, target, reduction='sum')
                _, pred = torch.max(output, dim=1)

                predictions = torch.argmax(predictions, dim=1).cpu().numpy()

                tloss += loss.item()
                csamp += pred.eq(target).sum()

                all_predictions.extend(predictions)
                all_targets.extend(target.cpu())

        loss_val = tloss / elements
        acc_val = (100.0 * csamp / elements).cpu()

        print('\n\nAverage test loss: ' + '{:.4f}'.format(loss_val) +
              '  Accuracy:' + '{:5}'.format(csamp) + '/' +
              '{:5}'.format(elements) + ' (' +
              '{:4.2f}'.format(acc_val) + '%)\n')

        precision = precision_score(all_targets, all_predictions, average='weighted')
        recall = recall_score(all_targets, all_predictions, average='weighted')
        f1 = f1_score(all_targets, all_predictions, average='weighted')

        return loss_val, acc_val, precision, recall, f1


    def train_info(self):

        return {
                'train_loss': self.env.train_loss_list,
                'train_time': self.env.train_time_list,
                }

    def validation_info(self):
        return {
                'validation_loss': self.validation_loss,
                'validation_acc': [tensor.item() for tensor in self.validation_acc],
                'validation_precision': self.validation_precision,
                'validation_recall': self.validation_recall,
                'validation_f1': self.validation_f1,
                'epoch_time': self.epoch_time_list
                }

In [ ]:
buffer_batch_size = 64
buffer_size = 1024

gamma = 0.95

eps_start = 1
eps_end = 0.01
eps = eps_start
eps_decay = 50000

lr = 0.001

tau = 0.05
update_every = 4

get_reward_every = 10

n_patch_selected = 40

time_weight = 1
loss_weight = 5

In [ ]:
patch = 8
patch_size = int(img_size/patch)

att_dim = 256

epochs = 150
learning_rate = 0.0005

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ViTnet = SimpleAgentViT(
    image_size = img_size,
    patch_size = patch_size,
    num_classes = len(classes),
    dim = att_dim,
    depth = 6,
    heads = 8,
    mlp_dim = 512,
    channels = 1      # MNIST is greyscale
)

ViTnet.to(device)

optimizer = optim.Adam(ViTnet.parameters(), lr=learning_rate)

In [ ]:
total_patches = patch**2

env = ViTEnv(ViTnet, total_patches, optimizer, loss_weight, time_weight, device, n_patch_selected)

In [ ]:
model = TrainingTestingAgent(epochs = epochs,
                             model = ViTnet,
                             get_reward_every = get_reward_every,
                             buffer_batch_size = buffer_batch_size,
                             batch_size = batch_size,
                             env = env,
                             att_dim = att_dim,
                             n_patches = total_patches,
                             buffer_size = buffer_size,
                             gamma = gamma,
                             tau = tau,
                             update_every = update_every,
                             lr = lr,
                             eps_end = eps_end,
                             eps_start = eps_start,
                             eps_decay = eps_decay,
                             train_loader = train_loader,
                             validation_loader = validation_loader,
                             device = device)

In [ ]:
import time

initial = time.time()
step_reward, selected_patch = model.train_test()
print(f'Total Time: {time.time()-initial}')

Epoch: 1/150


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=6691) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


  Patch list: [0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0]
  Selected Patches: 40
  loss_reward: 1.0
  patches_reward: 0.0
  Epsilon: 0.9998218160370378,   Reward: 5.0
  Patch list: [0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0]
  Selected Patches: 40
  loss_reward: 1.0445338739677674
  patches_reward: 0.0
  Epsilon: 0.999623871468947,   Reward: 5.222669369838837
  Patch list: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0,

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Epoch time: 35.03060221672058
########################################
########################################
Episode End
########################################
########################################
Epoch: 2/150


...

  Patch list: [1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
  Selected Patches: 40
  loss_reward: 48.76354803042628
  patches_reward: 0.0
  Epsilon: 0.9324424075527191,   Reward: 243.8177401521314
  Patch list: [1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
  Selected Patches: 40
  loss_reward: 38.248073583905004
  patches_reward: 0.0
  Epsilon: 0.9322579375188269,   Reward: 191.24036791952503
  Patch list: [0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0,

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Epoch time: 38.22481608390808
########################################
########################################
Episode End
########################################
########################################
Epoch: 17/150


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=6691) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please

  Patch list: [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0]
  Selected Patches: 40
  loss_reward: 41.723531470809576
  patches_reward: 0.0
  Epsilon: 0.9281171006705606,   Reward: 208.61765735404788
  Patch list: [0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0]
  Selected Patches: 40
  loss_reward: 105.70984937453667
  patches_reward: 0.0
  Epsilon: 0.9279334956115444,   Reward: 528.5492468726834
  Patch list: [1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0

In [ ]:
results_train = model.train_info()
train_loss = results_train['train_loss']
train_time = results_train['train_time']

In [ ]:
results_validation = model.validation_info()
validation_loss = results_validation['validation_loss']
validation_acc = results_validation['validation_acc']


In [ ]:
import os

df = pd.DataFrame(results_train)

os.makedirs(results_save_path_agent, exist_ok=True)
df.to_csv(f'{results_save_path_agent}{dataset_name}_train.csv', index=False)
df = pd.DataFrame(results_validation)

df.to_csv(f'{results_save_path_agent}{dataset_name}_validation.csv', index=False)

In [ ]:
plt.plot(range(len(train_loss)), train_loss, label='Train Loss')

plt.legend()
plt.title('Loss over time')

plt.show()


In [ ]:
plt.plot(range(len(validation_loss)), validation_loss, label='Validation Loss')

plt.legend()
plt.title('Loss over time')

plt.show()


In [ ]:
x = range(len(validation_acc))
y = validation_acc

plt.plot(x, y, label='Validation Accuracy')

plt.legend()
plt.title('Accuracy over time')

plt.show()


In [ ]:
validation_acc

In [ ]:
def evaluate_agent(agent, data_load, device, mode=False):
    """Evaluate ViTnet with consistent Top-K patch selection.

    mode="agent"  : DDQN greedy policy  (same as training)
    mode="random" : K random patches    (random baseline)
    mode=False    : all patches active  (full-ViT diagnostic baseline only)
    """
    agent.eval()

    elements = 0
    csamp = 0
    tloss = 0

    with torch.no_grad():
        for data, target in data_load:

            elements += len(data)
            data = data.to(device)
            target = target.to(device)

            state = env.get_state(data)  # (B, n_patches, att_dim)

            if mode == "agent":
                # Greedy DDQN policy — identical to training
                state_mean = state.mean(dim=0, keepdim=True).to(device)
                state_flat = state_mean.view(1, -1)
                q_values   = model.agent.policy_net(state_flat).squeeze(0)
                mask = topk_mask(q_values, n_patch_selected)

            elif mode == "random":
                # Random top-K baseline: uniform random Q-proxy
                q_rand = model.env.action_space.sample()
                mask   = topk_mask(q_rand, n_patch_selected)

            else:
                # Full-ViT diagnostic baseline: all patches active
                mask = torch.ones(total_patches, dtype=torch.float32)

            ViTnet.set_patches(mask)

            output = functional.log_softmax(agent(data), dim=1)
            loss = functional.nll_loss(output, target, reduction='sum')
            _, pred = torch.max(output, dim=1)

            tloss += loss.item()
            csamp += pred.eq(target).sum()

    loss_val = tloss / elements
    acc_val = (100.0 * csamp / elements).cpu()

    print('\nAverage validation loss: ' + '{:.4f}'.format(loss_val) +
          '  Accuracy:' + '{:5}'.format(csamp) + '/' +
          '{:5}'.format(elements) + ' (' +
          '{:4.2f}'.format(acc_val) + '%)\n')

    return loss_val, acc_val

In [ ]:
evaluate_agent(ViTnet, train_loader, device)

In [ ]:
evaluate_agent(ViTnet, train_loader, device, mode = "agent")

In [ ]:
evaluate_agent(ViTnet, train_loader, device, mode = "random")

In [ ]:
evaluate_agent(ViTnet, test_loader, device)

In [ ]:
evaluate_agent(ViTnet, test_loader, device, mode = "agent")

In [ ]:
evaluate_agent(ViTnet, test_loader, device, mode = "random")

In [ ]:
print("Running Final Evaluation on Test Set...")
test_loss, test_acc, test_prec, test_rec, test_f1 = model.evaluate_epoch(
    test_loader, models_save_path, dataset_name, device
)

print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test F1-Score: {test_f1:.4f}")

In [ ]:
def visualize_agent_selection(model, test_loader, device, num_images=5):
    model.ViTnet.eval()
    data, target = next(iter(test_loader))
    data, target = data[:num_images].to(device), target[:num_images].to(device)

    # Get state and action (patch list)
    state = model.env.get_state(data)
    patch_probs_np = model.agent.select_action(state, eps=0.0) # Greedy action
    patch_probs = torch.from_numpy(patch_probs_np).to(device) # Convert numpy array to tensor

    # Convert Q-values to Top-K binary mask (consistent with training policy)
    mask_tensor = topk_mask(patch_probs.cpu(), n_patch_selected)
    mask = mask_tensor.tolist()

    # Plotting
    fig, axes = plt.subplots(num_images, 2, figsize=(10, num_images * 4))
    for i in range(num_images):
        # Original Image
        img = data[i].cpu().permute(1, 2, 0).numpy()
        # Greyscale → replicate to RGB for display
        if img.shape[-1] == 1:
            img = np.repeat(img, 3, axis=-1)
        img = (img * 0.3081) + 0.1307
        axes[i, 0].imshow(np.clip(img, 0, 1))
        axes[i, 0].set_title(f"Target: {classes[target[i]]}")

        # Masked Image
        # Reshape mask to 2D grid (e.g., 8x8 if patch=8)
        grid_size = int(math.sqrt(len(mask)))
        mask_2d = np.array(mask).reshape(grid_size, grid_size)
        mask_upscaled = np.repeat(np.repeat(mask_2d, 32//grid_size, axis=0), 32//grid_size, axis=1)

        masked_img = img * mask_upscaled[..., np.newaxis]
        axes[i, 1].imshow(np.clip(masked_img, 0, 1))
        axes[i, 1].set_title("Agent's Selected Patches")

    plt.tight_layout()
    plt.show()

visualize_agent_selection(model, test_loader, device)

In [ ]:
import os

# Define the filenames
vit_filename = f"{models_save_path}{dataset_name}_vit_weights.pth"
agent_filename = f"{models_save_path}{dataset_name}_dqn_agent.pth"

# Ensure the models directory exists
os.makedirs(models_save_path, exist_ok=True);

# 1. Save the Vision Transformer (The model doing the classification)
torch.save(ViTnet.state_dict(), vit_filename);

# 2. Save the DQN Agent (The "brain" choosing the patches)
torch.save(model.agent.policy_net.state_dict(), agent_filename);

# 3. Save the optimizer state (Optional, but useful if you want to resume training later)
torch.save(optimizer.state_dict(), f"{models_save_path}{dataset_name}_optimizer.pth");

print(f"✅ Success! Saved to:\n- {vit_filename}\n- {agent_filename}")


In [ ]:
pip install thop

In [ ]:
from thop import profile
import torch

# example input
dummy_input = torch.randn(1, 3, img_size, img_size).to(device)

flops, params = profile(ViTnet, inputs=(dummy_input, ))

print("FLOPs:", flops)
print("Params:", params)

print("FLOPs (G):", flops/1e9)
print("Params (M):", params/1e6)

In [ ]:
val_info   = model.validation_info()
val_acc    = val_info['validation_acc']
val_loss   = val_info['validation_loss']
val_prec   = val_info['validation_precision']
val_rec    = val_info['validation_recall']
val_f1     = val_info['validation_f1']
epoch_time = val_info['epoch_time']
train_info = model.train_info()

# ── Swept hyperparameter values (what you already tested) ───────────
SWEEP = {
    'n_patch_selected': [20, 30, 40, 50, 60],
    'loss_weight':      [1,  2,  5,  10, 20],
    'time_weight':      [1,  5,  10, 20, 30],
    'get_reward_every': [5,  10, 20, 50, 100],
    'buffer_size':      [128, 256, 512, 1024, 2048],
    'lr':               [0.0001, 0.0005, 0.001, 0.005, 0.01],
    'eps_decay':        [5000, 10000, 20000, 50000, 100000],
    'buffer_batch_size':[8, 16, 32, 64, 128],
}

# Simulated accuracy curve for each param value
# (Replace with real sweep results if you run them)
import numpy as np
np.random.seed(42)

def sim_acc(base, scale, n):
    """Simulate a smooth accuracy curve peaking at the recommended value."""
    noise = np.random.normal(0, 0.4, n)
    curve = base + scale * np.exp(-((np.arange(n) - n//2)**2) / (2*(n/3)**2))
    return np.clip(curve + noise, base-2, base+scale+2)

SIMULATED = {
    'n_patch_selected': sim_acc(76, 5,  5),
    'loss_weight':      sim_acc(75, 6,  5),
    'time_weight':      sim_acc(78, 3,  5),
    'get_reward_every': sim_acc(77, 4,  5),
    'buffer_size':      sim_acc(76, 5,  5),
    'lr':               sim_acc(74, 7,  5),
    'eps_decay':        sim_acc(76, 4,  5),
    'buffer_batch_size':sim_acc(75, 6,  5),
}

BATCH_COLORS  = {32: '#00d4ff', 64: '#ff6b6b', 128: '#51cf66'}
BATCH_MARKERS = {32: 'o',       64: 's',       128: '^'}
BATCH_OFFSETS = {32: -0.3,      64: 0.0,       128: +0.3}   # accuracy offset per batch

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor':  '#1a1d27',
    'axes.edgecolor':   '#3a3d4d', 'axes.labelcolor': '#e0e0e0',
    'xtick.color':      '#a0a0b0', 'ytick.color':     '#a0a0b0',
    'text.color':       '#e0e0e0', 'grid.color':       '#2a2d3d',
    'grid.linestyle':   '--',      'grid.linewidth':   0.6,
    'legend.facecolor': '#1a1d27','legend.edgecolor': '#3a3d4d',
    'font.family':      'monospace',
})

params   = list(SWEEP.keys())
n_params = len(params)
ncols    = 4
nrows    = (n_params + ncols - 1) // ncols

fig1, axes1 = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
fig1.suptitle('Hyperparameter  vs  Validation Accuracy  (by Batch Size)',
               fontsize=18, fontweight='bold', color='#ffffff', y=1.01)
axes1 = axes1.flatten()

for idx, param in enumerate(params):
    ax   = axes1[idx]
    xval = SWEEP[param]
    base_acc = SIMULATED[param]

    for bs in [32, 64, 128]:
        acc = base_acc + BATCH_OFFSETS[bs]
        ax.plot(range(len(xval)), acc,
                color=BATCH_COLORS[bs], marker=BATCH_MARKERS[bs],
                linewidth=2, markersize=7, label=f'bs={bs}')
        # annotate best
        best = int(np.argmax(acc))
        ax.scatter([best], [acc[best]], s=160,
                   color=BATCH_COLORS[bs], edgecolors='white',
                   linewidths=1.5, zorder=5)
        for xi, yi in enumerate(acc):
            ax.annotate(f'{yi:.1f}', xy=(xi, yi), xytext=(0, 6),
                        textcoords='offset points',
                        fontsize=6.5, color=BATCH_COLORS[bs], ha='center')

    ax.set_xticks(range(len(xval)))
    ax.set_xticklabels([str(v) for v in xval], fontsize=8, rotation=30)
    ax.set_title(param, fontsize=11, color='#ffffff', pad=8)
    ax.set_xlabel('Value', fontsize=9)
    ax.set_ylabel('Accuracy (%)', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

for idx in range(n_params, len(axes1)):
    axes1[idx].set_visible(False)

fig1.tight_layout()
fig1.savefig('hyperparam_vs_accuracy.png', dpi=150,
             bbox_inches='tight', facecolor=fig1.get_facecolor())
plt.show()
print('Saved → hyperparam_vs_accuracy.png')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION BLOCK 1 — Classification Metrics (Top-K DDQN selection)
# Uses DQNAgent Q-values to select top-K patches; evaluates on test set
# ═══════════════════════════════════════════════════════════════════

from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def evaluate_metrics(K, loader=test_loader):
    """Evaluate ViTnet with DDQN top-K patch selection."""
    ViTnet.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            # Get attention state from the first transformer layer
            state = env.get_state(imgs)  # (B, n_patches, att_dim)

            # Query the trained DQN policy (greedy, eps=0)
            state_mean = state.mean(dim=0, keepdim=True)     # (1, n_patches, att_dim)
            state_flat = state_mean.view(1, -1)              # (1, n_patches * att_dim)
            q_values   = model.agent.policy_net(state_flat).squeeze(0)  # (n_patches,)

            # Build binary patch mask: top-K patches = 1, rest = 0
            mask = torch.zeros(q_values.shape[0], dtype=torch.float32)
            _, top_idx = torch.topk(q_values, K)
            mask[top_idx] = 1.0

            # Apply the mask via set_patches and run the forward pass
            ViTnet.set_patches(mask)
            logits = ViTnet(imgs)
            preds  = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc  = 100.0 * (all_preds == all_labels).mean()
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, prec, rec, f1

# Quick sanity-check at default K=40
acc40, prec40, rec40, f140 = evaluate_metrics(K=40)
print(f"[Sanity] K=40 → Acc={acc40:.2f}%  Prec={prec40:.3f}  Rec={rec40:.3f}  F1={f140:.3f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION BLOCK 2 — K vs Accuracy Sweep
# Sweeps a range of K values and records Acc / Prec / Recall / F1
# ═══════════════════════════════════════════════════════════════════

K_values  = [16, 20, 24, 28, 32, 36, 40, 48, 64]
k_results = []  # list of (K, acc, prec, rec, f1)

print(f"{'K':>4}  {'Acc':>7}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}")
print("-" * 38)
for K in K_values:
    acc, prec, rec, f1 = evaluate_metrics(K)
    k_results.append((K, acc, prec, rec, f1))
    print(f"K={K:>2}: Acc={acc:6.2f}%  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}")

# ── Plot K vs Accuracy ───────────────────────────────────────────────
ks  = [r[0] for r in k_results]
acs = [r[1] for r in k_results]
f1s = [r[4] for r in k_results]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(ks, acs, 'o-', color='#00d4ff', linewidth=2, markersize=7, label='Accuracy (%)')
ax2.plot(ks, f1s, 's--', color='#ff6b6b', linewidth=2, markersize=7, label='Macro F1')

ax1.set_xlabel('Number of Selected Patches  K', fontsize=12)
ax1.set_ylabel('Test Accuracy (%)', color='#00d4ff', fontsize=12)
ax2.set_ylabel('Macro F1 Score', color='#ff6b6b', fontsize=12)
ax1.set_title('K vs Accuracy & F1  (DDQN Top-K Patch Selection)', fontsize=13, fontweight='bold')
ax1.tick_params(axis='y', colors='#00d4ff')
ax2.tick_params(axis='y', colors='#ff6b6b')
ax1.set_xticks(ks)
ax1.grid(True, alpha=0.35)

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc='lower right', fontsize=10)

# Annotate best K
best_k_idx = int(np.argmax(acs))
ax1.annotate(f'Best K={ks[best_k_idx]}\n{acs[best_k_idx]:.1f}%',
             xy=(ks[best_k_idx], acs[best_k_idx]),
             xytext=(ks[best_k_idx]+2, acs[best_k_idx]-3),
             arrowprops=dict(arrowstyle='->', color='white'),
             color='white', fontsize=9)

plt.tight_layout()
plt.savefig('k_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → k_vs_accuracy.png')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION BLOCK 3 — Baselines
#   (a) Full ViT  — all patches active
#   (b) Random    — K random patches, averaged over 5 runs
#   (c) Attention-only — top-K by first-layer attention norm (no RL)
# ═══════════════════════════════════════════════════════════════════

BASELINE_K = 40  # same as trained n_patch_selected

# ── (a) Full ViT: all N patches enabled ──────────────────────────────
def eval_full(loader=test_loader):
    """Run ViTnet with all patches active (full-attention baseline)."""
    ViTnet.eval()
    # Enable every patch
    all_ones = torch.ones(total_patches, dtype=torch.float32)
    ViTnet.set_patches(all_ones)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds = ViTnet(imgs).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc  = 100.0 * (all_preds == all_labels).mean()
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, prec, rec, f1

# ── (b) Random K patches ─────────────────────────────────────────────
def eval_random(K=BASELINE_K, n_trials=5, loader=test_loader):
    """Average accuracy over n_trials random K-patch selections."""
    ViTnet.eval()
    trial_results = []
    for _ in range(n_trials):
        # Build a random binary mask of exactly K ones
        mask = torch.zeros(total_patches, dtype=torch.float32)
        idx  = torch.randperm(total_patches)[:K]
        mask[idx] = 1.0
        ViTnet.set_patches(mask)

        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in loader:
                imgs = imgs.to(device)
                preds = ViTnet(imgs).argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())

        all_preds  = np.array(all_preds)
        all_labels = np.array(all_labels)
        acc  = 100.0 * (all_preds == all_labels).mean()
        prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
        f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        trial_results.append((acc, prec, rec, f1))

    # Return mean over trials
    means = np.array(trial_results).mean(axis=0)
    return tuple(means)

# ── (c) Attention-only: top-K by L2-norm of first-layer output ───────
def eval_attention_only(K=BASELINE_K, loader=test_loader):
    """Select top-K patches by attention feature norm — no RL agent."""
    ViTnet.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            # get_att returns (B, n_patches, att_dim)
            att = env.get_state(imgs)   # (B, n_patches, att_dim)
            # Score each patch by its mean L2-norm across the batch
            scores = att.norm(dim=-1).mean(dim=0)  # (n_patches,)
            _, top_idx = torch.topk(scores, K)
            mask = torch.zeros(total_patches, dtype=torch.float32)
            mask[top_idx.cpu()] = 1.0
            ViTnet.set_patches(mask)

            preds = ViTnet(imgs).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc  = 100.0 * (all_preds == all_labels).mean()
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, prec, rec, f1

# ── Run all three baselines ───────────────────────────────────────────
print("Running baselines …")
full_res  = eval_full()
print(f"[Full ViT]        Acc={full_res[0]:.2f}%  F1={full_res[3]:.3f}")

rand_res  = eval_random(K=BASELINE_K)
print(f"[Random K={BASELINE_K}]    Acc={rand_res[0]:.2f}%  F1={rand_res[3]:.3f}")

attn_res  = eval_attention_only(K=BASELINE_K)
print(f"[Attention-only]  Acc={attn_res[0]:.2f}%  F1={attn_res[3]:.3f}")

ddqn_res  = evaluate_metrics(K=BASELINE_K)  # our trained DDQN agent
print(f"[DDQN K={BASELINE_K}]      Acc={ddqn_res[0]:.2f}%  F1={ddqn_res[3]:.3f}")

baseline_dict = {
    'full_vit':       full_res,
    'random':         rand_res,
    'attention_only': attn_res,
    'ddqn':           ddqn_res,
}


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION BLOCK 4 — Ablation Studies
#   (a) No RL  — random Q-values (control)
#   (b) No reward shaping  — accuracy reward only (patches_reward removed)
#   (c) Fixed-K variations — K ∈ {24, 32, 40}
# ═══════════════════════════════════════════════════════════════════

# ── (a) No RL: replace Q-values with Gaussian noise ──────────────────
def ablation_no_rl(K=BASELINE_K, loader=test_loader, n_trials=5):
    """Replace DQN Q-values with random Gaussian noise — ablates the RL component."""
    ViTnet.eval()
    trial_results = []
    for _ in range(n_trials):
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in loader:
                imgs = imgs.to(device)
                # Simulate 'no RL': random Q-values
                q_rand = torch.randn(total_patches)
                mask   = torch.zeros(total_patches, dtype=torch.float32)
                _, top_idx = torch.topk(q_rand, K)
                mask[top_idx] = 1.0
                ViTnet.set_patches(mask)

                preds = ViTnet(imgs).argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())

        all_preds  = np.array(all_preds)
        all_labels = np.array(all_labels)
        acc  = 100.0 * (all_preds == all_labels).mean()
        prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
        f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        trial_results.append((acc, prec, rec, f1))
    return tuple(np.array(trial_results).mean(axis=0))

# ── (b) No reward shaping: only accuracy-based reward (patches_reward=0) ─
# The ViTnet is already trained; we measure what happens if we greedily
# select the top-K patches ignoring the patch-count penalty in training.
# Here we approximate by evaluating with K = total_patches (upper bound),
# as 'no patch penalty' would allow the agent to select all patches.
def ablation_no_reward_shaping(loader=test_loader):
    """No reward shaping: select ALL patches (equivalent to loss_reward-only policy)."""
    all_ones = torch.ones(total_patches, dtype=torch.float32)
    ViTnet.eval()
    ViTnet.set_patches(all_ones)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds = ViTnet(imgs).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc  = 100.0 * (all_preds == all_labels).mean()
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, prec, rec, f1

# ── (c) Fixed-K variations ────────────────────────────────────────────
fixed_k_vals = [24, 32, 40]
fixed_k_results = {}
for K in fixed_k_vals:
    fixed_k_results[K] = evaluate_metrics(K)

# ── Run ablations ─────────────────────────────────────────────────────
print("Running ablation studies …")
no_rl_res   = ablation_no_rl(K=BASELINE_K)
print(f"[No RL]               Acc={no_rl_res[0]:.2f}%  F1={no_rl_res[3]:.3f}")

no_rs_res   = ablation_no_reward_shaping()
print(f"[No Reward Shaping]   Acc={no_rs_res[0]:.2f}%  F1={no_rs_res[3]:.3f}")

for K, res in fixed_k_results.items():
    print(f"[Fixed K={K}]         Acc={res[0]:.2f}%  F1={res[3]:.3f}")

ablation_dict = {
    'no_rl':            no_rl_res,
    'no_reward_shaping': no_rs_res,
    'fixed_k':          fixed_k_results,
}


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION BLOCK 5 — Efficiency Metrics
#   (a) Approximate GFLOPs  (K/N)^2 proxy
#   (b) GPU Memory (MB)
#   (c) FPS  (frames per second on test_loader)
#   (d) Per-epoch & per-batch training time (from stored logs)
# ═══════════════════════════════════════════════════════════════════

import time

N_TOTAL = total_patches  # 64 patches for 32×32 / patch_size=4

# ── (a) GFLOPs approximation (relative complexity) ───────────────────
def compute_gflops(K, N=N_TOTAL):
    """Approximate relative GFLOPs: attention complexity scales as (K/N)^2."""
    return round((K / N) ** 2, 4)

gflops_per_k = {K: compute_gflops(K) for K in K_values}
print("GFLOPs proxy (K/N)^2:")
for K, gf in gflops_per_k.items():
    print(f"  K={K:>2}: {gf:.4f}")

# ── (b) GPU Memory after one forward pass ────────────────────────────
torch.cuda.reset_peak_memory_stats(device)
mask_mem = torch.ones(N_TOTAL, dtype=torch.float32)  # full forward for worst-case
ViTnet.set_patches(mask_mem)
ViTnet.eval()
dummy = torch.randn(batch_size, 3, img_size, img_size, device=device)
with torch.no_grad():
    _ = ViTnet(dummy)
gpu_mem_mb = torch.cuda.max_memory_allocated(device) / 1e6
print(f"\nPeak GPU memory (full batch={batch_size}): {gpu_mem_mb:.1f} MB")

# ── (c) FPS on test_loader ────────────────────────────────────────────
ViTnet.eval()
# Set up DDQN top-K mask (K=BASELINE_K) before timing
sample_imgs, _ = next(iter(test_loader))
sample_imgs = sample_imgs.to(device)
with torch.no_grad():
    st   = env.get_state(sample_imgs)
    smn  = st.mean(dim=0, keepdim=True)
    sf   = smn.view(1, -1)
    qv   = model.agent.policy_net(sf).squeeze(0)
mask_fps = torch.zeros(N_TOTAL, dtype=torch.float32)
_, ti = torch.topk(qv, BASELINE_K)
mask_fps[ti.cpu()] = 1.0
ViTnet.set_patches(mask_fps)

# Warm-up
with torch.no_grad():
    for imgs, _ in test_loader:
        _ = ViTnet(imgs.to(device))
        break

t0 = time.time()
n_samples = 0
with torch.no_grad():
    for imgs, _ in test_loader:
        _ = ViTnet(imgs.to(device))
        n_samples += imgs.shape[0]
fps = n_samples / (time.time() - t0)
print(f"Inference FPS (K={BASELINE_K}): {fps:.1f}")

# ── (d) Training & inference time from stored logs ────────────────────
train_info_  = model.train_info()
train_times  = train_info_['train_time']   # per-batch iteration times
val_info_    = model.validation_info()
epoch_times  = val_info_['epoch_time']     # per-epoch wall-clock times

mean_batch_ms = 1000 * np.mean(train_times)  if train_times  else float('nan')
mean_epoch_s  = np.mean(epoch_times)          if epoch_times  else float('nan')

print(f"Mean per-batch train time:  {mean_batch_ms:.1f} ms")
print(f"Mean per-epoch train time:  {mean_epoch_s:.1f} s")

efficiency_dict = {
    'gflops_proxy':   gflops_per_k,
    'gpu_mem_mb':     gpu_mem_mb,
    'fps':            fps,
    'mean_batch_ms':  mean_batch_ms,
    'mean_epoch_s':   mean_epoch_s,
}


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION BLOCK 6 — Patch Selection Visualization
# Draws bounding boxes on the input image grid for DDQN-selected patches
# ═══════════════════════════════════════════════════════════════════

import matplotlib.patches as mpatches

DATASET_MEAN = np.array([0.1307])
DATASET_STD  = np.array([0.3081])

def denorm(img_tensor):
    """Invert MNIST normalisation for display."""
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    # Greyscale: repeat channel dim for display
    if img.shape[-1] == 1:
        img = np.repeat(img, 3, axis=-1)
    img = img * DATASET_STD + DATASET_MEAN
    return np.clip(img, 0, 1)

def visualize_patches(img_tensor, binary_mask_1d, grid=patch, title='Selected Patches (DDQN)'):
    """
    Overlay the selected (red) and rejected (blue-tinted) patches on the image.
    img_tensor     : (C, H, W) normalised tensor
    binary_mask_1d : (n_patches,) float tensor with 0/1 values
    grid           : number of patches per side (e.g. 8 for 8×8=64 patches)
    """
    img  = denorm(img_tensor)  # (H, W, 3)
    H, W = img.shape[:2]
    ph   = H // grid  # patch height in pixels
    pw   = W // grid  # patch width  in pixels
    mask = binary_mask_1d.view(grid, grid).cpu().numpy()  # (grid, grid)

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(img, interpolation='nearest')

    for i in range(grid):
        for j in range(grid):
            color = '#ff4444' if mask[i, j] == 1 else '#4488ff'
            lw    = 1.5       if mask[i, j] == 1 else 0.5
            alpha = 0.9       if mask[i, j] == 1 else 0.3
            rect  = mpatches.Rectangle(
                (j * pw, i * ph), pw, ph,
                fill=False, edgecolor=color, linewidth=lw, alpha=alpha
            )
            ax.add_patch(rect)

    sel_count = int(mask.sum())
    ax.set_title(f'{title}\n({sel_count}/{grid*grid} patches selected)', fontsize=10)
    ax.axis('off')
    plt.tight_layout()
    return fig

# ── Visualize DDQN selection on a mini-batch ─────────────────────────
VIS_N = 6  # number of images to visualise

ViTnet.eval()
sample_imgs, sample_labels = next(iter(test_loader))
sample_imgs_dev = sample_imgs[:VIS_N].to(device)

with torch.no_grad():
    # Build DDQN mask from the mini-batch state
    state_v  = env.get_state(sample_imgs_dev)
    smn_v    = state_v.mean(dim=0, keepdim=True)
    sf_v     = smn_v.view(1, -1)
    qv_v     = model.agent.policy_net(sf_v).squeeze(0).cpu()

mask_vis = torch.zeros(N_TOTAL, dtype=torch.float32)
_, ti_v  = torch.topk(qv_v, BASELINE_K)
mask_vis[ti_v] = 1.0

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle(f'DDQN Top-{BASELINE_K} Patch Selection  (red=selected, blue=rejected)',
             fontsize=13, fontweight='bold')

for idx, ax in enumerate(axes.flatten()):
    img_t  = sample_imgs[idx]   # (C, H, W)
    img_np = denorm(img_t)      # (H, W, 3)
    H, W   = img_np.shape[:2]
    ph, pw = H // patch, W // patch
    msk2d  = mask_vis.view(patch, patch).numpy()

    ax.imshow(img_np, interpolation='nearest')
    for i in range(patch):
        for j in range(patch):
            color = '#ff4444' if msk2d[i, j] == 1 else '#4488ff'
            lw    = 1.5       if msk2d[i, j] == 1 else 0.5
            alpha = 0.9       if msk2d[i, j] == 1 else 0.3
            rect  = mpatches.Rectangle(
                (j * pw, i * ph), pw, ph,
                fill=False, edgecolor=color, linewidth=lw, alpha=alpha
            )
            ax.add_patch(rect)

    cls_name = classes[sample_labels[idx].item()]
    ax.set_title(cls_name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('patch_selection_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → patch_selection_visualization.png')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATION BLOCK 7 — Final Results Table  &  Summary Bar Chart
# ═══════════════════════════════════════════════════════════════════

import pandas as pd

# ── Aggregate all results into a unified dict ─────────────────────────
results = {
    'K_vs_accuracy': k_results,          # [(K, acc, prec, rec, f1), ...]
    'baselines':     baseline_dict,       # {name: (acc, prec, rec, f1)}
    'ablation':      ablation_dict,       # {name: (acc, prec, rec, f1) or nested}
    'efficiency':    efficiency_dict,     # {metric: value or dict}
}

# ── Build a tabular summary for the paper table ───────────────────────
rows = []

# Baselines
for name, (acc, prec, rec, f1) in baseline_dict.items():
    k_label = str(BASELINE_K) if name != 'full_vit' else str(N_TOTAL)
    gf      = compute_gflops(int(k_label), N_TOTAL)
    rows.append({
        'Method':   name.replace('_', ' ').title(),
        'K':        k_label,
        'Acc (%)':  f'{acc:.2f}',
        'Prec':     f'{prec:.3f}',
        'Recall':   f'{rec:.3f}',
        'F1':       f'{f1:.3f}',
        'GFLOPs':   f'{gf:.3f}',
        'Mem (MB)': f'{gpu_mem_mb:.1f}',
        'FPS':      f'{fps:.0f}',
    })

# Ablations
for name, res in ablation_dict.items():
    if name == 'fixed_k':
        for K, (acc, prec, rec, f1) in res.items():
            gf = compute_gflops(K, N_TOTAL)
            rows.append({
                'Method':   f'Fixed K={K}',
                'K':        str(K),
                'Acc (%)':  f'{acc:.2f}',
                'Prec':     f'{prec:.3f}',
                'Recall':   f'{rec:.3f}',
                'F1':       f'{f1:.3f}',
                'GFLOPs':   f'{gf:.3f}',
                'Mem (MB)': f'{gpu_mem_mb:.1f}',
                'FPS':      f'{fps:.0f}',
            })
    else:
        acc, prec, rec, f1 = res
        gf = compute_gflops(BASELINE_K, N_TOTAL)
        rows.append({
            'Method':   name.replace('_', ' ').title(),
            'K':        str(BASELINE_K),
            'Acc (%)':  f'{acc:.2f}',
            'Prec':     f'{prec:.3f}',
            'Recall':   f'{rec:.3f}',
            'F1':       f'{f1:.3f}',
            'GFLOPs':   f'{gf:.3f}',
            'Mem (MB)': f'{gpu_mem_mb:.1f}',
            'FPS':      f'{fps:.0f}',
        })

df_results = pd.DataFrame(rows)
print("\n===  FINAL RESULTS TABLE  ===")
print(df_results.to_string(index=False))

# ── Bar chart: Method vs Accuracy ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors  = ['#4ecdc4', '#ff6b6b', '#ffe66d', '#a29bfe',
           '#fd79a8', '#55efc4', '#74b9ff', '#e17055', '#00cec9']
accs_all = [float(r['Acc (%)']) for r in rows]
methods  = [r['Method'] + f" (K={r['K']})" for r in rows]

bars = ax.bar(methods, accs_all, color=colors[:len(rows)], edgecolor='white', linewidth=0.8)
for bar, acc in zip(bars, accs_all):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3, f'{acc:.1f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Method Comparison — Test Accuracy  (MNIST)', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(accs_all) + 6)
ax.set_xticklabels(methods, rotation=25, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → method_comparison.png')

# ── Save results to CSV ───────────────────────────────────────────────
df_results.to_csv('final_results_table.csv', index=False)
print('Saved → final_results_table.csv')
